# Exploratory data analysis: Home Credit Default Risk

**Purpose:** check distributions, correlations, outliers and data risks on the **full
population**, and decide what this means for modeling (Phase 1, section II.4).

**Two backends, same queries:**

| `BACKEND` | Data | When |
|---|---|---|
| `"local"` | raw Kaggle CSVs loaded into DuckDB, with the **same cleaning rules as silver** (365243 → NULL + flag, XNA → NULL) | on a laptop, before the silver tables exist |
| `"databricks"` | `home_credit.silver.*` | final run for the submission (export the executed notebook as evidence) |

All statistics use **every row** unless a cell says *SAMPLE*. The only sample is a
systematic 5 % sample (`SK_ID_CURR % 20 = 0`), used for one scatter plot. It is
reproducible on both backends.

In [ ]:
# ---- configuration -------------------------------------------------------------
BACKEND = "local"            # "local" or "databricks"
DATA_DIR = "../data/raw/v1"  # local only: folder with the Kaggle CSVs
DUCKDB_FILE = "../data/eda.duckdb"
CATALOG, SCHEMA = "home_credit", "silver"
SAVE_FIGS = "../docs/eda"    # PNGs for the README; set to None to skip

import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
                     "axes.axisbelow": True, "axes.titleweight": "bold", "font.size": 9})
if SAVE_FIGS: os.makedirs(SAVE_FIGS, exist_ok=True)
def save(fig, name):
    if SAVE_FIGS: fig.savefig(f"{SAVE_FIGS}/{name}.png", bbox_inches="tight")

### Load / connect

In [ ]:
TABLES = {"application_train": "application_train.csv", "application_test": "application_test.csv",
          "bureau": "bureau.csv", "bureau_balance": "bureau_balance.csv",
          "previous_application": "previous_application.csv",
          "installments_payments": "installments_payments.csv",
          "pos_cash_balance": "POS_CASH_balance.csv", "credit_card_balance": "credit_card_balance.csv"}

if BACKEND == "local":
    import duckdb
    con = duckdb.connect(DUCKDB_FILE)
    existing = {r[0] for r in con.execute("SHOW TABLES").fetchall()}
    for t, f in TABLES.items():
        if t in existing:
            continue
        t0 = time.time()
        src = f"read_csv('{DATA_DIR}/{f}', header=true, sample_size=-1)"
        if t.startswith("application"):
            # mirror the silver cleaning rules so local and Databricks results match
            src = f"""(SELECT * REPLACE (
                        CASE WHEN DAYS_EMPLOYED = 365243 THEN NULL ELSE DAYS_EMPLOYED END AS DAYS_EMPLOYED,
                        NULLIF(CODE_GENDER, 'XNA') AS CODE_GENDER,
                        NULLIF(ORGANIZATION_TYPE, 'XNA') AS ORGANIZATION_TYPE,
                        NULLIF(NAME_FAMILY_STATUS, 'Unknown') AS NAME_FAMILY_STATUS),
                     CAST(DAYS_EMPLOYED = 365243 AS INT) AS DAYS_EMPLOYED_ANOM
                     FROM {src})"""
        con.execute(f"CREATE TABLE {t} AS SELECT * FROM {src}")
        print(f"loaded {t} in {time.time()-t0:.0f}s")
    def sql(q): return con.execute(q).df()
    def columns(t): return [r[0] for r in con.execute(f"DESCRIBE {t}").fetchall()]
else:
    spark.sql(f"USE CATALOG {CATALOG}"); spark.sql(f"USE SCHEMA {SCHEMA}")  # noqa: F821
    def sql(q): return spark.sql(q).toPandas()                              # noqa: F821
    def columns(t): return [c for c in spark.table(t).columns if not c.startswith("_")]  # noqa: F821
print("backend:", BACKEND)

## 1. Population counts

In [ ]:
keys = {"application_train": "SK_ID_CURR", "application_test": "SK_ID_CURR", "bureau": "SK_ID_CURR",
        "bureau_balance": "SK_ID_BUREAU", "previous_application": "SK_ID_CURR",
        "installments_payments": "SK_ID_CURR", "pos_cash_balance": "SK_ID_CURR",
        "credit_card_balance": "SK_ID_CURR"}
pop = pd.concat([sql(f"SELECT '{t}' AS table_name, count(*) AS n_rows, count(DISTINCT {k}) AS distinct_id, "
                     f"'{k}' AS id_column FROM {t}") for t, k in keys.items()], ignore_index=True)
pop["columns"] = [len(columns(t)) for t in keys]
pop["rows_per_id"] = pop["n_rows"] / pop["distinct_id"]
pop

## 2. Target: class imbalance

In [ ]:
tgt = sql("SELECT TARGET, count(*) AS n FROM application_train GROUP BY TARGET ORDER BY TARGET")
tgt["share"] = tgt.n / tgt.n.sum()
TARGET_RATE = float(tgt.loc[tgt.TARGET == 1, "share"].iloc[0])
print(f"default rate = {TARGET_RATE:.2%}  -> a constant 'repays' model scores {1-TARGET_RATE:.1%} accuracy")
tgt

## 3. Missing values (application_train)

In [ ]:
cols = [c for c in columns("application_train") if c not in ("SK_ID_CURR", "TARGET")]
exprs = ", ".join(f"avg(CASE WHEN `{c}` IS NULL THEN 1.0 ELSE 0.0 END) AS `{c}`" for c in cols)
if BACKEND == "local": exprs = exprs.replace("`", '"')
miss = sql(f"SELECT {exprs} FROM application_train").T[0].sort_values(ascending=False)
miss.name = "null_share"
print(f"{(miss > 0).sum()} of {len(miss)} columns have missing values; "
      f"{(miss > 0.5).sum()} are more than half empty")
top = miss.head(25)[::-1]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top.index, top.values, color=BLUE, height=0.7)
ax.set_xlim(0, 1); ax.set_xlabel("Share of rows missing")
ax.set_title("25 most incomplete columns in application_train (all rows)", loc="left")
for y, v in enumerate(top.values): ax.text(v + .01, y, f"{v:.0%}", va="center", color=MUTED, fontsize=7)
save(fig, "missing_values"); plt.show()

In [ ]:
# Is missingness itself informative? Default rate when the value is missing vs present.
def rate_by_missing(col):
    c = f'"{col}"' if BACKEND == "local" else f"`{col}`"
    return sql(f"""SELECT '{col}' AS col,
                    avg(CASE WHEN {c} IS NULL THEN TARGET END) AS default_rate_missing,
                    avg(CASE WHEN {c} IS NOT NULL THEN TARGET END) AS default_rate_present
                    FROM application_train""")
pd.concat([rate_by_missing(c) for c in ["EXT_SOURCE_1", "EXT_SOURCE_3", "OWN_CAR_AGE",
                                        "OCCUPATION_TYPE", "AMT_REQ_CREDIT_BUREAU_YEAR"]
           if c in cols], ignore_index=True)

## 4. Distributions and outliers

In [ ]:
num = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "DAYS_BIRTH",
       "DAYS_EMPLOYED", "DAYS_REGISTRATION", "CNT_CHILDREN", "CNT_FAM_MEMBERS",
       "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
def pct(c, p): return f"percentile_cont({p}) WITHIN GROUP (ORDER BY {c})"
rows = []
for c in num:
    r = sql(f"""SELECT '{c}' AS col, count({c}) AS n, min({c}) AS min, {pct(c, .01)} AS p01,
               {pct(c, .5)} AS median, {pct(c, .99)} AS p99, max({c}) AS max FROM application_train""")
    rows.append(r)
q = pd.concat(rows, ignore_index=True).set_index("col")
q["max_over_p99"] = q["max"] / q["p99"]
q

In [ ]:
# Extreme values: rows far beyond the 99th percentile (IQR rule on the log scale for money columns)
out = sql(f"""
WITH s AS (SELECT {pct('ln(AMT_INCOME_TOTAL)', .25)} AS q1, {pct('ln(AMT_INCOME_TOTAL)', .75)} AS q3 FROM application_train)
SELECT count(*) AS income_outliers_log_iqr,
       max(AMT_INCOME_TOTAL) AS max_income,
       avg(TARGET) AS default_rate_of_outliers
FROM application_train, s
WHERE ln(AMT_INCOME_TOTAL) > q3 + 3 * (q3 - q1) OR ln(AMT_INCOME_TOTAL) < q1 - 3 * (q3 - q1)""")
anom = sql("""SELECT DAYS_EMPLOYED_ANOM, count(*) AS n, avg(TARGET) AS default_rate
               FROM application_train GROUP BY 1 ORDER BY 1""")
display(out); display(anom)

In [ ]:
# Histograms from binned SQL counts (full population, no sampling)
def hist(expr, lo, hi, bins, where="TRUE"):
    w = (hi - lo) / bins
    h = sql(f"""SELECT floor(({expr} - {lo}) / {w}) AS b, count(*) AS n FROM application_train
               WHERE {expr} IS NOT NULL AND {expr} >= {lo} AND {expr} < {hi} AND {where} GROUP BY 1 ORDER BY 1""")
    return lo + h.b * w, h.n, w

panels = [("ln(AMT_INCOME_TOTAL)", 10, 14, 40, "log income"),
          ("ln(AMT_CREDIT)", 10, 15, 40, "log credit amount"),
          ("-DAYS_BIRTH / 365.25", 20, 70, 50, "age (years)"),
          ("-DAYS_EMPLOYED / 365.25", 0, 45, 45, "years employed (placeholder removed)")]
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
for ax, (e, lo, hi, b, title) in zip(axes, panels):
    x, n, w = hist(e, lo, hi, b)
    ax.bar(x, n, width=w * 0.9, align="edge", color=BLUE)
    ax.set_title(title, loc="left"); ax.set_ylabel("applications" if ax is axes[0] else "")
fig.suptitle("Distributions of key application variables (all rows)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); save(fig, "distributions"); plt.show()

## 5. Default rate by category

In [ ]:
def rate_by(col, min_n=1000):
    return sql(f"""SELECT coalesce(CAST({col} AS STRING), '(missing)') AS level, count(*) AS n, avg(TARGET) AS default_rate
                  FROM application_train GROUP BY 1 HAVING count(*) >= {min_n} ORDER BY default_rate DESC""")
cats = ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "CODE_GENDER", "NAME_CONTRACT_TYPE",
        "NAME_FAMILY_STATUS", "OCCUPATION_TYPE"]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, c in zip(axes.flat, cats):
    r = rate_by(c)[::-1]
    ax.barh(r.level.str.slice(0, 28), r.default_rate, color=BLUE, height=0.7)
    ax.axvline(TARGET_RATE, color=MUTED, lw=1, ls="--")
    ax.set_title(c, loc="left"); ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
fig.suptitle(f"Default rate by category (levels with >= 1,000 applications; dashed = overall {TARGET_RATE:.1%})",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); save(fig, "default_rate_by_category"); plt.show()

## 6. Relationship with the target

In [ ]:
# Pearson correlation with TARGET (point-biserial), full population
dtypes = (con.execute("DESCRIBE application_train").df().set_index("column_name")["column_type"]
          if BACKEND == "local" else pd.Series(dict(spark.table("application_train").dtypes)))  # noqa: F821
numeric = [c for c in columns("application_train") if c not in ("SK_ID_CURR", "TARGET")
           and str(dtypes[c]).lower() in ("double", "bigint", "integer", "int", "float", "smallint", "tinyint")]
q_ = lambda c: f'"{c}"' if BACKEND == "local" else f"`{c}`"
corr = sql("SELECT " + ", ".join(f"corr({q_(c)}, TARGET) AS {q_(c)}" for c in numeric) + " FROM application_train").T[0]
corr = corr.dropna().sort_values(key=abs, ascending=False)
top = corr.head(15)[::-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(top.index, top.values, color=[BLUE if v < 0 else ORANGE for v in top.values], height=0.7)
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_xlabel("Pearson correlation with TARGET")
ax.set_title("15 strongest linear correlations with default (all rows)\nblue = lower risk as value rises, orange = higher risk", loc="left")
save(fig, "correlation_with_target"); plt.show()
corr.head(15).to_frame("corr_with_target")

In [ ]:
# Correlation among the main numeric inputs (multicollinearity check), full population
main = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "DAYS_BIRTH",
        "DAYS_EMPLOYED", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "CNT_CHILDREN", "CNT_FAM_MEMBERS"]
pairs = [(a, b) for i, a in enumerate(main) for b in main[i + 1:]]
r = sql("SELECT " + ", ".join(f"corr({a}, {b}) AS c{i}" for i, (a, b) in enumerate(pairs)) + " FROM application_train").iloc[0]
M = pd.DataFrame(np.eye(len(main)), index=main, columns=main)
for i, (a, b) in enumerate(pairs): M.loc[a, b] = M.loc[b, a] = r[f"c{i}"]
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(M.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(main)), main, rotation=60, ha="right"); ax.set_yticks(range(len(main)), main)
for i in range(len(main)):
    for j in range(len(main)):
        ax.text(j, i, f"{M.values[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(M.values[i, j]) > .6 else INK)
ax.grid(False); fig.colorbar(im, shrink=.7, label="Pearson r")
ax.set_title("Correlation between main numeric inputs (all rows)", loc="left")
save(fig, "correlation_matrix"); plt.show()
strong = [(a, b, round(r[f'c{i}'], 3)) for i, (a, b) in enumerate(pairs) if abs(r[f"c{i}"]) > .8]
print("pairs with |r| > 0.8:", strong)

In [ ]:
# Default rate by decile: is the relationship monotonic? (full population)
def decile_rate(expr):
    return sql(f"""SELECT d, avg(TARGET) AS default_rate, count(*) AS n FROM (
                    SELECT TARGET, ntile(10) OVER (ORDER BY {expr}) AS d FROM application_train
                    WHERE {expr} IS NOT NULL) x GROUP BY d ORDER BY d""")
specs = [("EXT_SOURCE_2", "EXT_SOURCE_2"), ("EXT_SOURCE_3", "EXT_SOURCE_3"),
         ("AMT_CREDIT / AMT_INCOME_TOTAL", "credit / income"), ("-DAYS_BIRTH", "age")]
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for ax, (e, t) in zip(axes, specs):
    d = decile_rate(e)
    ax.bar(d.d, d.default_rate, color=BLUE, width=.8)
    ax.axhline(TARGET_RATE, color=MUTED, lw=1, ls="--"); ax.set_title(t, loc="left")
    ax.set_xticks(range(1, 11)); ax.set_xlabel("decile (low -> high)")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
fig.suptitle("Default rate by decile (all rows)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); save(fig, "default_rate_by_decile"); plt.show()

In [ ]:
# SAMPLE: systematic 5 % sample (SK_ID_CURR % 20 = 0), only for this scatter plot
s = sql("""SELECT AMT_INCOME_TOTAL, AMT_CREDIT, TARGET FROM application_train
           WHERE SK_ID_CURR % 20 = 0 AND AMT_INCOME_TOTAL < 1000000""")
print(f"sample size: {len(s):,} of {int(pop.loc[pop.table_name=='application_train','n_rows'].iloc[0]):,} rows "
      f"(incomes >= 1,000,000 excluded from the plot only)")
fig, ax = plt.subplots(figsize=(6.5, 4))
for t, col, lab in [(0, BLUE, "repaid (0)"), (1, ORANGE, "default (1)")]:
    p = s[s.TARGET == t]
    ax.scatter(p.AMT_INCOME_TOTAL, p.AMT_CREDIT, s=4, alpha=.35 if t == 0 else .6, color=col, label=lab, linewidths=0)
ax.set_xlabel("AMT_INCOME_TOTAL"); ax.set_ylabel("AMT_CREDIT"); ax.legend(frameon=False, markerscale=3)
ax.set_title("Credit vs income (5 % systematic SAMPLE)", loc="left")
save(fig, "credit_vs_income_sample"); plt.show()

## 7. History tables: coverage and signal for the hypothesis

The hypothesis says borrowing history adds predictive power. Two checks before building
features: (a) how many applicants have any history, and (b) whether a simple history
measure separates defaulters.

In [ ]:
cov = sql("""
SELECT count(*) AS applicants,
  avg(CASE WHEN b.id IS NOT NULL THEN 1.0 ELSE 0 END) AS has_bureau,
  avg(CASE WHEN p.id IS NOT NULL THEN 1.0 ELSE 0 END) AS has_previous_application,
  avg(CASE WHEN i.id IS NOT NULL THEN 1.0 ELSE 0 END) AS has_installments,
  avg(CASE WHEN c.id IS NOT NULL THEN 1.0 ELSE 0 END) AS has_credit_card,
  avg(CASE WHEN b.id IS NULL AND p.id IS NULL THEN 1.0 ELSE 0 END) AS no_history_at_all
FROM application_train a
LEFT JOIN (SELECT DISTINCT SK_ID_CURR id FROM bureau) b ON a.SK_ID_CURR = b.id
LEFT JOIN (SELECT DISTINCT SK_ID_CURR id FROM previous_application) p ON a.SK_ID_CURR = p.id
LEFT JOIN (SELECT DISTINCT SK_ID_CURR id FROM installments_payments) i ON a.SK_ID_CURR = i.id
LEFT JOIN (SELECT DISTINCT SK_ID_CURR id FROM credit_card_balance) c ON a.SK_ID_CURR = c.id""")
cov.T.rename(columns={0: "value"})

In [ ]:
# Late-payment rate on past Home Credit installments vs default (full population).
# Payments are first grouped per installment (one installment can be paid in several rows).
late = sql("""
WITH inst AS (
  SELECT SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER,
         min(DAYS_INSTALMENT) AS due_day, max(DAYS_ENTRY_PAYMENT) AS pay_day
  FROM installments_payments WHERE DAYS_INSTALMENT <= 0 GROUP BY 1, 2, 3, 4),
per_app AS (
  SELECT SK_ID_CURR, avg(CASE WHEN pay_day > due_day THEN 1.0 ELSE 0.0 END) AS late_rate
  FROM inst GROUP BY 1)
SELECT CASE WHEN p.late_rate IS NULL THEN '0 no installment history'
            WHEN p.late_rate = 0 THEN '1 never late'
            WHEN p.late_rate <= 0.05 THEN '2 late <= 5%'
            WHEN p.late_rate <= 0.20 THEN '3 late 5-20%'
            ELSE '4 late > 20%' END AS bucket,
       count(*) AS applicants, avg(a.TARGET) AS default_rate
FROM application_train a LEFT JOIN per_app p ON a.SK_ID_CURR = p.SK_ID_CURR
GROUP BY 1 ORDER BY 1""")
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(late.bucket.str[2:], late.default_rate, color=BLUE, width=.6)
ax.axhline(TARGET_RATE, color=MUTED, lw=1, ls="--")
for x, (v, n) in enumerate(zip(late.default_rate, late.applicants)):
    ax.text(x, v, f"{v:.1%}\nn={n:,}", ha="center", va="bottom", fontsize=7, color=MUTED)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Default rate by past late-payment rate (all rows)", loc="left")
save(fig, "default_rate_by_late_payments"); plt.show()
late

In [ ]:
# Data risks in the history tables (full population)
risks = sql("""
SELECT 'bureau_balance rows whose SK_ID_BUREAU is not in bureau' AS check_name,
       (SELECT count(*) FROM bureau_balance bb WHERE NOT EXISTS (SELECT 1 FROM bureau b WHERE b.SK_ID_BUREAU = bb.SK_ID_BUREAU)) AS n
UNION ALL SELECT 'previous_application with 365243 in DAYS_FIRST_DUE (cleaned in silver)',
       (SELECT count(*) FROM previous_application WHERE DAYS_FIRST_DUE = 365243)
UNION ALL SELECT 'installments with missing payment (DAYS_ENTRY_PAYMENT NULL)',
       (SELECT count(*) FROM installments_payments WHERE DAYS_ENTRY_PAYMENT IS NULL)
UNION ALL SELECT 'bureau credits with negative debt',
       (SELECT count(*) FROM bureau WHERE AMT_CREDIT_SUM_DEBT < 0)
UNION ALL SELECT 'credit card months with zero limit',
       (SELECT count(*) FROM credit_card_balance WHERE AMT_CREDIT_LIMIT_ACTUAL = 0)""")
risks

## 8. Summary for modeling readiness

In [ ]:
lines = [
  f"* Population: {int(pop.n_rows.sum()):,} rows across {len(pop)} tables; application_train has "
  f"{int(pop.loc[pop.table_name=='application_train','n_rows'].iloc[0]):,} labelled applications.",
  f"* Target imbalance: {TARGET_RATE:.2%} defaults -> use ROC-AUC and recall at a fixed approval rate, "
  f"stratified split; never accuracy.",
  f"* Missing values: {(miss > 0).sum()} of {len(miss)} application columns have NULLs, {(miss > .5).sum()} are >50% empty "
  f"(EXT_SOURCE_1 {miss.get('EXT_SOURCE_1', float('nan')):.0%}); missingness is kept as NULL and imputed in training only.",
  f"* DAYS_EMPLOYED placeholder: {int(anom.loc[anom.DAYS_EMPLOYED_ANOM==1,'n'].sum()):,} rows "
  f"(default rate {anom.loc[anom.DAYS_EMPLOYED_ANOM==1,'default_rate'].mean():.2%} vs "
  f"{anom.loc[anom.DAYS_EMPLOYED_ANOM==0,'default_rate'].mean():.2%}) -> NULL + DAYS_EMPLOYED_ANOM flag.",
  f"* Outliers: max income is {q.loc['AMT_INCOME_TOTAL','max_over_p99']:.0f}x its 99th percentile "
  f"({int(out.income_outliers_log_iqr.iloc[0])} extreme rows) -> use ratios / log scale; tree models are robust.",
  f"* Strongest single signals: " + ", ".join(f"{k} ({v:+.3f})" for k, v in corr.head(4).items()) + ".",
  f"* Collinear pairs (|r|>0.8): {', '.join(f'{a}~{b}' for a, b, _ in strong) or 'none'} -> matters for the logistic baseline.",
  f"* History coverage: {cov.has_bureau.iloc[0]:.0%} have bureau records, {cov.has_previous_application.iloc[0]:.0%} "
  f"previous applications; {cov.no_history_at_all.iloc[0]:.0%} have neither (history features are NULL/0 for them).",
  f"* Hypothesis signal: default rate is {late.default_rate.iloc[1]:.1%} for applicants never late vs "
  f"{late.default_rate.iloc[-1]:.1%} for those late on > 20% of installments.",
]
from IPython.display import Markdown
Markdown("\n".join(lines))

**Interpretation (fill in after the full-data run):** add 3-5 sentences on what the numbers
above mean for Phase 2, e.g. which features to trust, whether the EXT_SOURCE scores are
risky (see the leakage notes in `docs/feature_spec.md`), and whether the history signal is
strong enough to expect the 0.03 AUC gain.